# Curve Bootstrapping, Swap Pricing & Sensitivity Analysis

This notebook:
1. Bootstraps **SOFR**, **TermSOFR3m**, **ICP**, and **CLP/USD collateral** curves
2. Prices a **5Y SOFR IRS**, a **TermSOFR3m basis swap**, and a **CLP ICP OIS swap**
3. Computes the **CLP/USD cross-currency basis** from the collateral curve
4. Computes **DV01 / sensitivity** per pillar by bumping exposed `SimpleQuote` objects

In [1]:
import json
from curveengine import CurveEngine
import ORE as ore
import pandas as pd

## Load Configuration & Bootstrap All Curves

Market data loaded from `sensitivity_config.json`.

In [2]:
with open("sensitivity_config.json") as f:
    config = json.load(f)

engine = CurveEngine(config)

# Retrieve curves and indexes
curve_names = ["SOFR", "TermSOFR3m", "ICP", "Collateral(CLP/USD)"]
curves = {}
for name in curve_names:
    curves[name] = {
        'curve': engine.getCurve(name),
        'index': engine.getIndex(name),
        'quotes': engine.getQuotes(name)
    }

refDate = ore.Date(11, 11, 2025)
ore.Settings.instance().evaluationDate = refDate
print(f"Curves built   : SOFR, TermSOFR3m, ICP, Collateral(CLP/USD)")

Curves built   : SOFR, TermSOFR3m, ICP, Collateral(CLP/USD)


In [3]:
with open('qs_results.json') as file:
    qs_results = json.load(file)

In [4]:
def qs_sensitivities_for(product_label):
    """Return a dict  pillar_name -> dv01  for the given QS product."""
    for p in qs_results['products']:
        if p['label'] == product_label:
            return {s['pillar']: s['dv01'] for s in p['sensitivities']}
    return {}

def classify_pillar(pillar):
    """Return (curve_bucket, tenor) for a QS pillar name."""
    if pillar == 'USD/CLP':
        return ('FX', 'USD/CLP')
    parts = pillar.rsplit('_', 1)
    tenor = parts[-1] if len(parts) > 1 else pillar
    if 'TermSOFR3m' in pillar:
        return ('TermSOFR3m', tenor)
    if 'FxForwardPoints' in pillar or 'CrossCurrencySwap' in pillar:
        return ('CLP_COLLUSD', tenor)
    if 'ICP' in pillar:
        return ('ICP', tenor)
    if 'SOFR' in pillar:
        return ('SOFR', tenor)
    return ('Other', tenor)

def build_qs_lookup(qs_sens):
    """Build dict (curve_bucket, tenor) -> dv01 from QS sensitivities."""
    lookup = {}
    for pillar, dv01 in qs_sens.items():
        curve, tenor = classify_pillar(pillar)
        lookup[(curve, tenor)] = dv01
    return lookup

def compare_sensitivities(ore_rows, qs_label, fx_divisor=1.0):
    """Merge ORE bump DV01s with QS AD DV01s and return a comparison DataFrame."""
    qs_sens = qs_sensitivities_for(qs_label)
    qs_lookup = build_qs_lookup(qs_sens)
    for row in ore_rows:
        curve = row.get('Curve', row.get('curve_bucket', ''))
        tenor = row['Tenor']
        qs_dv01 = qs_lookup.get((curve, tenor), 0.0)
        row['DV01_QS'] = round(qs_dv01, 4)
        row['Diff'] = round(row['DV01 (USD/bp)'] - qs_dv01, 4)
    return pd.DataFrame(ore_rows)

In [5]:
ore_data = {}
dc = ore.Actual360()
for key,vals in curves.items():
    curve = vals['curve']
    nodes = curve.nodes()
    ore_data[name] = {
        'date': [d.ISO() for d,_ in nodes],
        'yfs':[dc.yearFraction(refDate, d) for d,_ in nodes],
        'dfs': [d for _,d in nodes]
    }

qs_data = {}
dc = ore.Actual360()
for curve in qs_results['curves']:
    nodes = curve['nodes']
    qs_data[curve['name'] ] = {
        'date': [d['date'] for d in nodes],
        'yfs':[d['year_fraction'] for d in nodes],
        'dfs': [d['discount_factor'] for d in nodes],
    }

compare = {}
for k in ore_data.keys():
    ore_df = pd.DataFrame(ore_data[k])
    qs_df = pd.DataFrame(qs_data[k])
    compare[k] = ore_df.merge(qs_df, right_on='date', left_on='date', how='outer', suffixes=['_ore','_qs'])
    compare[k]['diff'] = compare[k]['dfs_ore'] - compare[k]['dfs_qs']

In [6]:
for k in compare.keys():
    print(f"Curve: {k}")
    display(compare[k])

Curve: Collateral(CLP/USD)


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000e+00
1,2025-12-11,0.083333,0.994817,0.083333,0.994817,-2.731149e-14
2,2026-02-11,0.255556,0.983468,0.255556,0.983468,0.000000e+00
3,2026-05-11,0.502778,0.965857,0.502778,0.965857,-6.772360e-15
4,2026-11-11,1.013889,0.950459,1.013889,0.950459,5.873080e-14
5,2027-11-11,2.027778,0.907731,2.027778,0.907731,3.219647e-15
6,2028-11-11,3.044444,0.869967,3.044444,0.869967,9.214851e-15
7,2030-11-11,5.072222,0.796764,5.072222,0.796764,4.773959e-15
8,2032-11-11,7.102778,0.730045,7.102778,0.730045,-5.639933e-14
9,2035-11-11,10.144444,0.645087,10.144444,0.645087,1.587619e-14


In [7]:
calendar   = ore.NullCalendar()
convention = ore.Following
notional   = 10_000_000.0
start      = calendar.advance(refDate, ore.Period("2D"))
maturity   = calendar.advance(start, ore.Period("5Y"))

sofr_curve = curves['SOFR']['curve']
sofr_index = curves['SOFR']['index']
fixed_sched = ore.Schedule(start, maturity, ore.Period(ore.Semiannual),
                           calendar, convention, convention,
                           ore.DateGeneration.Forward, False)
float_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                           calendar, convention, convention,
                           ore.DateGeneration.Forward, False)

sofr_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, notional,
                             fixed_sched, 0.0378, dc,
                             float_sched, sofr_index, 0.0, dc)

sofr_handle = ore.YieldTermStructureHandle(sofr_curve)
sofr_swap.setPricingEngine(ore.DiscountingSwapEngine(sofr_handle))

print(f"SOFR OIS 5Y  |  NPV = {sofr_swap.NPV():,.2f} USD  |  Fair Rate = {sofr_swap.fairRate():.6%}")

# # Cashflows as DataFrame
# rows = []
# for leg_name, leg_cfs, cast_fn in [("Fixed", sofr_swap.fixedLeg(), ore.as_fixed_rate_coupon),
#                                     ("Float", sofr_swap.floatingLeg(), ore.as_floating_rate_coupon)]:
#     for cf in leg_cfs:
#         cpn = cast_fn(cf)
#         rows.append({
#             "Leg": leg_name,
#             "PayDate": str(cf.date()),
#             "AccStart": str(cpn.accrualStartDate()) if cpn else "",
#             "AccEnd": str(cpn.accrualEndDate()) if cpn else "",
#             "YearFrac": round(cpn.accrualPeriod(), 6) if cpn else 0.0,
#             "Amount": round(cf.amount(), 2),
#             "DF": round(sofr_curve.discount(cf.date()), 10),
#         })

# pd.DataFrame(rows)

SOFR OIS 5Y  |  NPV = 342.60 USD  |  Fair Rate = 3.779250%


In [8]:
sofr_tenors  = ["1D","3M","6M","1Y","2Y","3Y","4Y","5Y","7Y","10Y","15Y","20Y","30Y"]
sofr_quotes  = curves['SOFR']['quotes']
bump = 1e-4
base_npv = sofr_swap.NPV()
sofr_swap_rows = []
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = sofr_swap.NPV() - base_npv
    rq.setValue(orig)
    sofr_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "Quote": round(orig, 6), "DV01 (USD/bp)": round(dv01, 2)})
compare_sensitivities(sofr_swap_rows, "SOFR OIS 5Y Swap")

,Curve,Tenor,Quote,DV01 (USD/bp),DV01_QS,Diff
0,SOFR,1D,0.04330,2.75,2.7463,0.0037
1,SOFR,3M,0.04335,2.78,2.7769,0.0031
2,SOFR,6M,0.04250,0.10,0.1015,-0.0015
3,SOFR,1Y,0.04100,0.00,0.0030,-0.0030
4,SOFR,2Y,0.03920,-0.00,-0.0006,0.0006
5,SOFR,3Y,0.03840,-0.00,-0.0007,0.0007
6,SOFR,4Y,0.03800,-0.00,-0.0009,0.0009
7,SOFR,5Y,0.03780,-4554.84,-4555.1788,0.3388
8,SOFR,7Y,0.03770,-17.69,-17.6898,-0.0002
9,SOFR,10Y,0.03790,-0.00,0.0000,-0.0000


In [9]:
ts3m_index = curves['TermSOFR3m']['index']
ts3m_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, notional,
                             fixed_sched, 0.04, dc,
                             float_sched, ts3m_index, 0.0, dc)
ts3m_swap.setPricingEngine(ore.DiscountingSwapEngine(sofr_handle))

print(f"TermSOFR3m 5Y  |  NPV = {ts3m_swap.NPV():,.2f} USD  |  Fair Rate = {ts3m_swap.fairRate():.6%}")

TermSOFR3m 5Y  |  NPV = 88,754.68 USD  |  Fair Rate = 3.805675%


In [10]:
ts3m_tenors  = ["3M","6M","1Y","2Y","3Y","4Y","5Y","7Y","10Y","15Y","20Y","30Y"]
ts3m_quotes  = curves['TermSOFR3m']['quotes']
base_npv_ts  = ts3m_swap.NPV()

ts3m_swap_rows = []

# Bump TermSOFR3m (forecast curve)
for tenor, q in zip(ts3m_tenors, ts3m_quotes):
    rq = q.get('spread', q.get('rate'))
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = ts3m_swap.NPV() - base_npv_ts
    rq.setValue(orig)
    ts3m_swap_rows.append({"Curve": "TermSOFR3m", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (discount curve)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = ts3m_swap.NPV() - base_npv_ts
    rq.setValue(orig)
    ts3m_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

compare_sensitivities(ts3m_swap_rows, "TermSOFR3m 5Y Swap")

,Curve,Tenor,DV01 (USD/bp),DV01_QS,Diff
0,TermSOFR3m,3M,5.56,5.5605,-0.0005
1,TermSOFR3m,6M,-0.01,-0.0112,0.0012
2,TermSOFR3m,1Y,0.00,0.0021,-0.0021
3,TermSOFR3m,2Y,-0.00,-0.0043,0.0043
4,TermSOFR3m,3Y,0.00,-0.0022,0.0022
5,TermSOFR3m,4Y,0.01,-0.0011,0.0111
6,TermSOFR3m,5Y,-4576.88,-4576.8884,0.0084
7,TermSOFR3m,7Y,-17.69,-17.6894,-0.0006
8,TermSOFR3m,10Y,-0.00,0.0000,-0.0000
9,TermSOFR3m,15Y,-0.00,0.0000,-0.0000


In [11]:
clp_notional = 5_000_000_000.0
clp_fixed_rate = 0.0440

icp_index = curves['ICP']['index']
coll_curve = curves['Collateral(CLP/USD)']['curve']
clp_fixed_sched = ore.Schedule(start, maturity, ore.Period(ore.Semiannual),
                               calendar, convention, convention,
                               ore.DateGeneration.Forward, False)
clp_float_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                               calendar, convention, convention,
                               ore.DateGeneration.Forward, False)

icp_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, clp_notional,
                            clp_fixed_sched, clp_fixed_rate, dc,
                            clp_float_sched, icp_index, 0.0, dc)

# Discount with CLP/USD collateral curve
coll_handle = ore.YieldTermStructureHandle(coll_curve)
icp_swap.setPricingEngine(ore.DiscountingSwapEngine(coll_handle))

print(f"CLP ICP OIS 5Y  |  NPV = {icp_swap.NPV():,.2f} CLP")

CLP ICP OIS 5Y  |  NPV = 40,241.56 CLP


In [12]:
fx_spot = 935.0
icp_tenors  = ["1D","3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y"]
icp_quotes  = curves['ICP']['quotes']
coll_tenors = ["1M","3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y"]
coll_quotes = curves['Collateral(CLP/USD)']['quotes']

base_npv_icp = icp_swap.NPV()
icp_swap_rows = []

# Bump ICP (forecast curve)
for tenor, q in zip(icp_tenors, icp_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = (icp_swap.NPV() - base_npv_icp) / fx_spot
    rq.setValue(orig)
    icp_swap_rows.append({"Curve": "ICP", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump CLP_COLLUSD (collateral / discount curve)
for tenor, q in zip(coll_tenors, coll_quotes):
    if 'fxPoints' in q:
        rq = q['fxPoints']
    elif 'rate' in q:
        rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = (icp_swap.NPV() - base_npv_icp) / fx_spot
    rq.setValue(orig)
    icp_swap_rows.append({"Curve": "CLP_COLLUSD", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (affects collateral curve via xccy helpers)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = (icp_swap.NPV() - base_npv_icp) / fx_spot
    rq.setValue(orig)
    icp_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

compare_sensitivities(icp_swap_rows, "CLP ICP OIS 5Y Swap (USD collateral)")

,Curve,Tenor,DV01 (USD/bp),DV01_QS,Diff
0,ICP,1D,1.46,1.4624,-0.0024
1,ICP,3M,-0.94,1.5648,-2.5048
2,ICP,6M,2.45,-0.0745,2.5245
3,ICP,1Y,-0.12,0.0258,-0.1458
4,ICP,2Y,-0.74,-0.0028,-0.7372
5,ICP,3Y,0.50,-0.0017,0.5017
6,ICP,5Y,-2360.84,-2381.7572,20.9172
7,ICP,7Y,-9.10,-9.2183,0.1183
8,ICP,10Y,-0.00,0.0000,-0.0000
9,ICP,20Y,-0.00,0.0000,-0.0000


In [13]:
fx_spot     = 935.0
usd_not     = 10_000_000.0
clp_not     = usd_not * fx_spot
xccy_spread = 0.0050  # 50 bp on CLP (receive) leg

icp_curve = curves['ICP']['curve']
xccy_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                          calendar, convention, convention,
                          ore.DateGeneration.Forward, False)

# Pay USD SOFR flat, Receive CLP ICP + 50 bp
xccy_swap = ore.CrossCcyBasisSwap(
    usd_not,  ore.USDCurrency(), xccy_sched, sofr_index, 0.0, 1.0,
    clp_not,  ore.CLPCurrency(), xccy_sched, icp_index, xccy_spread, 1.0
)

coll_handle = ore.YieldTermStructureHandle(coll_curve)
xccy_engine = ore.CrossCcySwapEngine(
    ore.USDCurrency(), sofr_handle,
    ore.CLPCurrency(), coll_handle,
    ore.QuoteHandle(ore.SimpleQuote(1/fx_spot))
)
xccy_swap.setPricingEngine(xccy_engine)

print(f"CLP/USD XCCY Basis 5Y  |  NPV = {xccy_swap.NPV():,.2f}")
print(f"  Pay USD leg NPV = {xccy_swap.legNPV(0):,.2f}")
print(f"  Rec CLP leg NPV = {xccy_swap.legNPV(1):,.2f}")
print(f"  Fair Spread on CLP leg = {xccy_swap.fairPaySpread()*10000:.2f} bp")

CLP/USD XCCY Basis 5Y  |  NPV = 158,589.08
  Pay USD leg NPV = 0.00
  Rec CLP leg NPV = 158,589.08
  Fair Spread on CLP leg = 34.56 bp


In [14]:
base_npv_xccy = xccy_swap.NPV()
xccy_swap_rows = []

# Bump ICP (CLP forecast curve)
for tenor, q in zip(icp_tenors, icp_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = xccy_swap.NPV() - base_npv_xccy
    rq.setValue(orig)
    xccy_swap_rows.append({"Curve": "ICP", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump CLP_COLLUSD (collateral / discount for CLP leg)
for tenor, q in zip(coll_tenors, coll_quotes):
    if 'fxPoints' in q:
        rq = q['fxPoints']
    elif 'rate' in q:
        rq = q['rate']
    else:
        continue
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = xccy_swap.NPV() - base_npv_xccy
    rq.setValue(orig)
    xccy_swap_rows.append({"Curve": "CLP_COLLUSD", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (USD discount)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = xccy_swap.NPV() - base_npv_xccy
    rq.setValue(orig)
    xccy_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

compare_sensitivities(xccy_swap_rows, "Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR)")

,Curve,Tenor,DV01 (USD/bp),DV01_QS,Diff
0,ICP,1D,-2.73,-2.7347,0.0047
1,ICP,3M,1.76,-2.9262,4.6862
2,ICP,6M,-4.59,0.1393,-4.7293
3,ICP,1Y,0.23,-0.0482,0.2782
4,ICP,2Y,1.39,0.0053,1.3847
5,ICP,3Y,-0.94,0.0032,-0.9432
6,ICP,5Y,4414.77,4453.8859,-39.1159
7,ICP,7Y,17.01,17.2383,-0.2283
8,ICP,10Y,-0.00,0.0000,-0.0000
9,ICP,20Y,-0.00,0.0000,-0.0000
